# 13 · Regression Evaluation Metrics

## What this notebook covers
This series has been classifier-focused. Regression models require a different metric family. This notebook covers the standard suite — from the familiar RMSE to the less-known but important quantile loss and interval coverage metrics.

## The metric taxonomy
| Metric | Sensitive to | Use when |
|--------|-------------|---------|
| **MAE** | All errors equally | Outliers shouldn't dominate |
| **RMSE** | Large errors (squared) | Large errors are especially costly |
| **MAPE** | Relative errors | Values span orders of magnitude |
| **sMAPE** | Symmetric relative | MAPE denominator instability |
| **R²** | Explained variance | Comparing across datasets |
| **Quantile loss (pinball)** | Asymmetric cost | Demand forecasting, risk |
| **CRPS** | Full predictive distribution | Probabilistic forecasts |
| **Coverage** | Prediction interval validity | Any interval forecast |

## The MAPE trap
MAPE = mean |y - ŷ| / |y| explodes when y ≈ 0 and is asymmetric (over-forecasting penalised differently than under-forecasting). For intermittent demand or near-zero targets, use sMAPE or RMSSE instead.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import QuantileRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

X, y = make_regression(n_samples=1000, n_features=20, n_informative=10, noise=30, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

clf = GradientBoostingRegressor(n_estimators=150, random_state=0)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

In [ ]:
def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mape(y_true, y_pred, eps=1e-8):
    return np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))) * 100

def smape(y_true, y_pred):
    """Symmetric MAPE — bounded [0, 200%], avoids MAPE denominator issues."""
    return np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100

def r2(y_true, y_pred):
    return r2_score(y_true, y_pred)

metrics = {
    "MAE"  : mae(y_te, y_pred),
    "RMSE" : rmse(y_te, y_pred),
    "MAPE" : mape(y_te, y_pred),
    "sMAPE": smape(y_te, y_pred),
    "R²"   : r2(y_te, y_pred),
}
for k, v in metrics.items():
    print(f"  {k:<8}: {v:.4f}")

In [ ]:
def quantile_loss(y_true, y_pred, alpha):
    """Pinball loss for quantile alpha ∈ (0, 1)."""
    err = y_true - y_pred
    return np.mean(np.where(err >= 0, alpha * err, (alpha - 1) * err))

# Train quantile regressors for lower (10th) and upper (90th) prediction intervals
q10 = QuantileRegressor(quantile=0.10, alpha=0.01, solver="highs")
q50 = QuantileRegressor(quantile=0.50, alpha=0.01, solver="highs")
q90 = QuantileRegressor(quantile=0.90, alpha=0.01, solver="highs")

q10.fit(X_tr, y_tr); q50.fit(X_tr, y_tr); q90.fit(X_tr, y_tr)
pred_10 = q10.predict(X_te)
pred_50 = q50.predict(X_te)
pred_90 = q90.predict(X_te)

print(f"Pinball loss @ q=0.10: {quantile_loss(y_te, pred_10, 0.10):.4f}")
print(f"Pinball loss @ q=0.50: {quantile_loss(y_te, pred_50, 0.50):.4f}")
print(f"Pinball loss @ q=0.90: {quantile_loss(y_te, pred_90, 0.90):.4f}")

# Coverage: fraction of true values inside [q10, q90] prediction interval
coverage = np.mean((y_te >= pred_10) & (y_te <= pred_90))
avg_width = np.mean(pred_90 - pred_10)
print(f"\n80% prediction interval:")
print(f"  Coverage (target 80%): {coverage:.2%}")
print(f"  Average width        : {avg_width:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Predicted vs actual
axes[0].scatter(y_te, y_pred, alpha=0.3, s=10, color="steelblue")
lim = [y_te.min(), y_te.max()]
axes[0].plot(lim, lim, "r--", lw=1.5, label="Perfect")
axes[0].set_xlabel("Actual"); axes[0].set_ylabel("Predicted")
axes[0].set_title(f"Predicted vs Actual  (R²={r2(y_te, y_pred):.3f})")
axes[0].legend()

# Residual distribution
residuals = y_te - y_pred
axes[1].hist(residuals, bins=40, color="steelblue", alpha=0.8, edgecolor="white")
axes[1].axvline(0, color="red", lw=1.5, linestyle="--")
axes[1].set_xlabel("Residual"); axes[1].set_title(f"Residuals  (MAE={mae(y_te, y_pred):.2f}, RMSE={rmse(y_te, y_pred):.2f})")

# Prediction interval on sorted sample
sort_idx = np.argsort(y_te)[:100]
x_plot = np.arange(100)
axes[2].fill_between(x_plot, pred_10[sort_idx], pred_90[sort_idx], alpha=0.3, color="steelblue", label="80% PI")
axes[2].scatter(x_plot, y_te[sort_idx], s=10, color="red", zorder=3, label="Actual")
axes[2].plot(x_plot, pred_50[sort_idx], color="steelblue", lw=1.5, label="Median")
axes[2].set_title(f"Prediction intervals (coverage={coverage:.0%})")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{base}/13_regression_metrics.png", dpi=120)
plt.show()